## 1 — Leitura das bases e inspeção de colunas

Carrega `workspace.default.gold_features_clientes`, centraliza os imports e inspeciona o esquema e as primeiras linhas antes do treinamento.

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Ajuste o nome da tabela conforme a base atual de clientes
df_clientes = spark.table("workspace.default.gold_features_clientes")
df_clientes.printSchema()
display(df_clientes.limit(5))

## 2 — Etapa 1: Baseline do modelo (v1)

Treina o **Isolation Forest** (`n_estimators=100`, `contamination=0.05`, `random_state=42`) com `StandardScaler` sobre as variáveis de cadastro e e-mail para estabelecer `flag_v1` e `score_v1`.

In [0]:
# Substitua pelas colunas numéricas reais de clientes da tabela
col_id = "id_cliente"
features_v1 = [
    "tamanho_prefixo_email",
    "is_email_temporario",
    "categoria_provedor_num",
    "hora_cadastro",
    "cadastro_fim_de_semana",
    "flag_atualizacao_anterior_cadastro",
]

pdf_base = df_clientes.select([col_id] + features_v1).dropna().toPandas()

scaler_v1 = StandardScaler()
X_v1 = scaler_v1.fit_transform(pdf_base[features_v1])

iso_v1 = IsolationForest(
    n_estimators=100, contamination=0.05, random_state=42
)
pdf_base["flag_v1"] = iso_v1.fit_predict(X_v1)  # -1 = anomalia, 1 = normal
pdf_base["score_v1"] = iso_v1.decision_function(X_v1)

print("Baseline v1 — contagem por flag (-1 anomalia / 1 normal):")
print(pdf_base["flag_v1"].value_counts())

## 3 — Etapa 2: Evolução do baseline (v2)

Adiciona `cadastro_madrugada` (`hora_cadastro < 6`), ajusta `n_estimators=300` e `contamination=0.03` e consolida os resultados (`flag_v2`, `score_v2`) para comparação com a v1.

In [0]:
pdf_evo = pdf_base[[col_id] + features_v1].copy()

# Feature derivada: cadastro na madrugada (0h às 5h)
pdf_evo["cadastro_madrugada"] = (pdf_evo["hora_cadastro"] < 6).astype(int)
features_v2 = features_v1 + ["cadastro_madrugada"]

scaler_v2 = StandardScaler()
X_v2 = scaler_v2.fit_transform(pdf_evo[features_v2])

iso_v2 = IsolationForest(
    n_estimators=300,
    contamination=0.03,
    random_state=42,
)
pdf_evo["flag_v2"] = iso_v2.fit_predict(X_v2)  # -1 = anomalia, 1 = normal
pdf_evo["score_v2"] = iso_v2.decision_function(X_v2)

df_comp = pdf_base[[col_id, "flag_v1", "score_v1"]].merge(
    pdf_evo, on=col_id, how="inner"
)

print("Evolução v2 — contagem por flag (-1 anomalia / 1 normal):")
print(pdf_evo["flag_v2"].value_counts())

## 4 — Comparação quantitativa e exemplos concretos

Compara v1 e v2 em volume, taxa de anomalias, scores médios e transições de classe, exibindo os maiores desvios na v2 e casos que deixaram de ser marcados como anomalia.

In [0]:
# 1) Resumo comparativo v1 vs. v2
resumo = pd.DataFrame({
    "qtd_anomalias_v1": [(df_comp["flag_v1"] == -1).sum()],
    "qtd_anomalias_v2": [(df_comp["flag_v2"] == -1).sum()],
    "taxa_anomalia_v1_%": [round((df_comp["flag_v1"] == -1).mean() * 100, 2)],
    "taxa_anomalia_v2_%": [round((df_comp["flag_v2"] == -1).mean() * 100, 2)],
    "score_medio_v1": [round(df_comp["score_v1"].mean(), 4)],
    "score_medio_v2": [round(df_comp["score_v2"].mean(), 4)],
    "mantiveram_anomalia": [((df_comp["flag_v1"] == -1) & (df_comp["flag_v2"] == -1)).sum()],
    "deixaram_anomalia_v2": [((df_comp["flag_v1"] == -1) & (df_comp["flag_v2"] == 1)).sum()],
    "novas_anomalias_v2": [((df_comp["flag_v1"] == 1) & (df_comp["flag_v2"] == -1)).sum()],
})
display(spark.createDataFrame(resumo))

# 2) Casos mais anômalos no v2 (menor score_v2 = maior desvio)
top_anomalos = df_comp[df_comp["flag_v2"] == -1].nsmallest(5, "score_v2")
print("=== TOP ANOMALIAS (EVOLUÇÃO V2) ===")
display(spark.createDataFrame(top_anomalos))

# 3) Registros que passaram de anomalia (v1=-1) para normal (v2=1)
ajuste_ruido = df_comp[(df_comp["flag_v1"] == -1) & (df_comp["flag_v2"] == 1)].head(3)
print("=== EXEMPLO DE AJUSTE DE RUÍDO (V1=-1 -> V2=1) ===")
display(spark.createDataFrame(ajuste_ruido))

## 5 — Conclusão

* **Baseline v1**: 363 clientes avaliados (344 normais e 19 anomalias / 5,23%).
* **Evolução v2**: 11 anomalias (3,03%), focando desvios mais extremos e reduzindo alertas de menor relevância.